# Attention U-Net

In [ ]:
import random

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision.datasets import OxfordIIITPet
from torchvision.transforms import InterpolationMode
import torchvision.transforms.functional as TF

try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
image_size = (512, 384)
batch_size = 1
num_epochs = 10
learning_rate = 1e-4

torch.manual_seed(42)
random.seed(42)

print("device:", device)


## Data

In [ ]:
train_dataset = OxfordIIITPet(
    root="U-Nets/data",
    split="trainval",
    target_types="segmentation",
    download=False,
)

test_dataset = OxfordIIITPet(
    root="U-Nets/data",
    split="test",
    target_types="segmentation",
    download=False,
)


In [ ]:
target_size = (500, 375)
keep_ratio = 0.20

matching_indices = []

for idx in range(len(train_dataset)):
    image, _ = train_dataset[idx]

    if image.size == target_size:
        matching_indices.append(idx)

num_keep = int(len(matching_indices) * keep_ratio)
selected_indices = random.sample(matching_indices, num_keep)
train_subset = Subset(train_dataset, selected_indices)

print("matching images:", len(matching_indices))
print("selected images:", len(train_subset))


In [ ]:
class PetSegmentationDataset(Dataset):
    def __init__(self, dataset, image_size=(512, 384)):
        self.dataset = dataset
        self.image_size = image_size

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        image, mask = self.dataset[idx]

        image = TF.resize(image, self.image_size, interpolation=InterpolationMode.BILINEAR)
        mask = TF.resize(mask, self.image_size, interpolation=InterpolationMode.NEAREST)

        image = TF.to_tensor(image)
        mask = torch.as_tensor(np.array(mask), dtype=torch.long)
        mask = (mask != 2).float().unsqueeze(0)

        return image, mask


train_seg_dataset = PetSegmentationDataset(train_subset, image_size=image_size)
test_seg_dataset = PetSegmentationDataset(test_dataset, image_size=image_size)

train_loader = DataLoader(train_seg_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_seg_dataset, batch_size=batch_size, shuffle=False)

images, masks = next(iter(train_loader))
print("images:", images.shape)
print("masks:", masks.shape)
print("mask values:", masks.unique())


## Model

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.conv(x)


In [ ]:
class Encoder(nn.Module):
    def __init__(self):
        super().__init__()

        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        self.down1 = DoubleConv(3, 64)
        self.down2 = DoubleConv(64, 128)
        self.down3 = DoubleConv(128, 256)
        self.down4 = DoubleConv(256, 512)

        self.bottleneck = DoubleConv(512, 1024)

    def forward(self, x):
        skip1 = self.down1(x)
        x = self.pool(skip1)

        skip2 = self.down2(x)
        x = self.pool(skip2)

        skip3 = self.down3(x)
        x = self.pool(skip3)

        skip4 = self.down4(x)
        x = self.pool(skip4)

        latent = self.bottleneck(x)

        return latent, skip1, skip2, skip3, skip4


In [ ]:
class AttentionGate(nn.Module):
    def __init__(self, skip_channels, gating_channels, hidden_channels):
        super().__init__()

        self.skip_projection = nn.Conv2d(skip_channels, hidden_channels, kernel_size=1)
        self.gating_projection = nn.Conv2d(gating_channels, hidden_channels, kernel_size=1)
        self.attention = nn.Sequential(
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden_channels, 1, kernel_size=1),
            nn.Sigmoid(),
        )

    def forward(self, skip, gating):
        if gating.shape[-2:] != skip.shape[-2:]:
            gating = F.interpolate(gating, size=skip.shape[-2:], mode="bilinear", align_corners=False)

        skip_features = self.skip_projection(skip)
        gating_features = self.gating_projection(gating)
        alpha = self.attention(skip_features + gating_features)

        return skip * alpha, alpha


In [ ]:
class AttentionUpBlock(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels):
        super().__init__()

        self.up = nn.ConvTranspose2d(in_channels, out_channels, kernel_size=2, stride=2)
        self.attention_gate = AttentionGate(skip_channels, out_channels, out_channels // 2)
        self.conv = DoubleConv(out_channels + skip_channels, out_channels)

    def forward(self, x, skip):
        x = self.up(x)
        filtered_skip, attention_map = self.attention_gate(skip, x)
        x = torch.cat([filtered_skip, x], dim=1)
        x = self.conv(x)

        return x, attention_map


In [ ]:
class AttentionDecoder(nn.Module):
    def __init__(self):
        super().__init__()

        self.up1 = AttentionUpBlock(1024, 512, 512)
        self.up2 = AttentionUpBlock(512, 256, 256)
        self.up3 = AttentionUpBlock(256, 128, 128)
        self.up4 = AttentionUpBlock(128, 64, 64)

        self.final_conv = nn.Conv2d(64, 1, kernel_size=1)

    def forward(self, latent, skip1, skip2, skip3, skip4):
        x, attention4 = self.up1(latent, skip4)
        x, attention3 = self.up2(x, skip3)
        x, attention2 = self.up3(x, skip2)
        x, attention1 = self.up4(x, skip1)

        logits = self.final_conv(x)
        attention_maps = [attention1, attention2, attention3, attention4]

        return logits, attention_maps


In [ ]:
class AttentionUNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.encoder = Encoder()
        self.decoder = AttentionDecoder()

    def forward(self, x, return_attention=False):
        latent, skip1, skip2, skip3, skip4 = self.encoder(x)
        logits, attention_maps = self.decoder(latent, skip1, skip2, skip3, skip4)

        if return_attention:
            return logits, attention_maps

        return logits


In [ ]:
model = AttentionUNet().to(device)

x = torch.randn(1, 3, 512, 384, device=device)

with torch.no_grad():
    logits, attention_maps = model(x, return_attention=True)

print("logits:", logits.shape)

for idx, attention_map in enumerate(attention_maps, start=1):
    print(f"attention{idx}:", attention_map.shape)


## Loss and training

In [ ]:
loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)


def dice_loss_from_logits(logits, masks, eps=1e-7):
    probs = torch.sigmoid(logits)

    intersection = (probs * masks).sum(dim=(1, 2, 3))
    probs_sum = probs.sum(dim=(1, 2, 3))
    masks_sum = masks.sum(dim=(1, 2, 3))

    dice = (2 * intersection + eps) / (probs_sum + masks_sum + eps)

    return 1 - dice.mean()


In [ ]:
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    total_bce_loss = 0
    total_dice_loss = 0

    for images, masks in train_loader:
        images = images.to(device)
        masks = masks.to(device).float()

        logits = model(images)

        bce_loss = loss_fn(logits, masks)
        dice_loss = dice_loss_from_logits(logits, masks)
        loss = bce_loss + dice_loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_bce_loss += bce_loss.item()
        total_dice_loss += dice_loss.item()

    avg_loss = total_loss / len(train_loader)
    avg_bce_loss = total_bce_loss / len(train_loader)
    avg_dice_loss = total_dice_loss / len(train_loader)

    print(
        f"Epoch {epoch + 1}/{num_epochs}, "
        f"Loss: {avg_loss:.4f}, "
        f"BCE: {avg_bce_loss:.4f}, "
        f"Dice: {avg_dice_loss:.4f}"
    )


## Test

In [ ]:
def dice_score(pred_mask, true_mask, eps=1e-7):
    pred_mask = pred_mask.float()
    true_mask = true_mask.float()

    intersection = (pred_mask * true_mask).sum(dim=(1, 2, 3))
    pred_sum = pred_mask.sum(dim=(1, 2, 3))
    true_sum = true_mask.sum(dim=(1, 2, 3))

    dice = (2 * intersection + eps) / (pred_sum + true_sum + eps)
    return dice.mean()


def iou_score(pred_mask, true_mask, eps=1e-7):
    pred_mask = pred_mask.float()
    true_mask = true_mask.float()

    intersection = (pred_mask * true_mask).sum(dim=(1, 2, 3))
    union = pred_mask.sum(dim=(1, 2, 3)) + true_mask.sum(dim=(1, 2, 3)) - intersection

    iou = (intersection + eps) / (union + eps)
    return iou.mean()


In [ ]:
model.eval()

images, masks = next(iter(test_loader))
images = images.to(device)
masks = masks.to(device).float()

with torch.no_grad():
    logits, attention_maps = model(images, return_attention=True)
    probs = torch.sigmoid(logits)
    pred_masks = (probs > 0.5).float()

print("images:", images.shape)
print("logits:", logits.shape)
print("probability range:", probs.min().item(), probs.max().item())
print("dice:", dice_score(pred_masks, masks).item())
print("iou:", iou_score(pred_masks, masks).item())


In [ ]:
if plt is None:
    print("matplotlib is not installed, so plotting is skipped.")
else:
    image_to_show = images[0].detach().cpu().permute(1, 2, 0)
    true_mask_to_show = masks[0, 0].detach().cpu()
    prob_to_show = probs[0, 0].detach().cpu()
    pred_mask_to_show = pred_masks[0, 0].detach().cpu()
    attention_to_show = attention_maps[0][0, 0].detach().cpu()

    plt.figure(figsize=(16, 4))

    plt.subplot(1, 5, 1)
    plt.imshow(image_to_show)
    plt.title("Image")
    plt.axis("off")

    plt.subplot(1, 5, 2)
    plt.imshow(true_mask_to_show, cmap="gray")
    plt.title("True mask")
    plt.axis("off")

    plt.subplot(1, 5, 3)
    plt.imshow(prob_to_show, cmap="gray")
    plt.title("Probability")
    plt.axis("off")

    plt.subplot(1, 5, 4)
    plt.imshow(pred_mask_to_show, cmap="gray")
    plt.title("Pred mask")
    plt.axis("off")

    plt.subplot(1, 5, 5)
    plt.imshow(attention_to_show, cmap="gray")
    plt.title("Attention")
    plt.axis("off")

    plt.show()
